# **Atividade Prática**
<font size=3>

- **Tema:** autocomplete.
- **Prazo de entrega:** 15 de Junho.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

## **Enunciado:**
<font size=3>

Vamos realizar uma tarefa de **_autocomplete_** com o conjundo de dados [IMDB movie reviews](https://keras.io/api/datasets/imdb/), também disponível no diretório $\text{dataset/}\,$. Nesta tarefa, iremos encontrar o próximo *token* mais provável. Para isso, realize os seguintes passos atentamente:

### **1º Passo:**
<font size=3>

- Importe o *dataset* e realize uma amostragem dos dados a fim de capturar 500 textos de revisão;
- Defina os dados textuais em uma lista.


In [1]:
#from google.colab import drive
#drive.mount('/content/drive/')

In [ ]:
from collections import Counter
from keras import layers, Model
import keras_tuner as kt
from keras.metrics import SparseTopKCategoricalAccuracy
from keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib.ticker import FuncFormatter
import os
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorboard
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from tensorflow.keras.preprocessing.sequence import pad_sequences

RANDOM_STATE = 42

In [3]:
print(tf.config.list_physical_devices())
print(tf.config.list_physical_devices("GPU"))
#tf.debugging.set_log_device_placement(True)

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
df = pd.read_csv('./dataset/imdb.csv', ).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(df.shape)
df.head()

(50000, 2)


,reviews,label
0,River's Edge is more than just the story of a ...,1
1,Evening is the beautiful story of the flawed l...,1
2,I question anyone saying they don't care for t...,1
3,This docu-drama is what you would expect from ...,1
4,"As several posters have ""hinted,"" this is a so...",0


### **2º Passo:**
<font size=3>

- Realize a indexação dos textos, utilizando a a função [`TextVectorization`](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/):
    - Considere, *inicialmente*, um vocabulário de tamanho 5000;
    - Faça a indexação sem preenchimento (*padding*). Para isso, defina as variáveis `output_sequence_length = None` e `ragged = True`.
    <br>

- Defina a lista de vocabulário por meio do método `get_vocabulary` do objeto de `TextVectorization`;

- Defina os dados indexados com o nome `token_ids`.


In [5]:
df = df.sample( n = 500 )

df.head()

,reviews,label
46743,The atmosphere in this show is great. There's ...,0
30774,I wanted to see Valentine ever since I saw tha...,1
28589,"If this is classed as 'real life' of London, t...",0
30996,Full House was and still is a great show. It's...,1
15504,Divorced single mom in picturesque seaside tow...,0


In [6]:
texts = df['reviews'].tolist()
texts[:5]

['The atmosphere in this show is great. There\'s plenty of excellent buildup, but thats where this show fails. There\'s way to much build up for nothing. You will constantly see a creepy set up that makes it feel likes something really freaky is coming right out of the corner and then....nothing. Over and over again nothing. You hear plenty of stories of people talking about freaky events but you see none. They show up at these peoples doors, talk about their deep and emotional pasts, set up lame equipment and find nothing! there is nothing on this show thats leads me to believe in anything paranormal. I laugh every time they need to exercise a "horrible spirit" that we as an audience have seen nothing of. They get rid of the spirit that never was and everything is put in a neat little package. A show that looked so freaky and had such great potential leads up to one thing...Nothing!',
 "I wanted to see Valentine ever since I saw that Denise Richards and Marley Shelton were in it becau

### **3º Passo:**
<font size=3>

- O *array* `token_ids` é formado por vetores indexados de tamanhos distintos. Mas, agora, iremos decompor cada vetor, *e.g.* $[2,\, 5,\, 3,\, 1,\, 8]$, como sequências do "próximo *token*":
$$
    [2,\, 5],\, [2,\, 5,\, 3],\, [2,\, 5,\, 3,\, 1],\, [2,\, 5,\, 3,\, 1,\, 8]
$$

- Assim, defina uma lista `sequences` que irá capturar as sequências de **todos** os vetores;

- Utilize a função [`pad_sequences`]() para preencher as sequências da esqueda para a direira:
$$
    [ 0,\, 0,\, 0,\, 2,\, 5],\, [0,\, 0,\, 2,\, 5,\, 3],\, [0,\, 2,\, 5,\, 3,\, 1],\, [2,\, 5,\, 3,\, 1,\, 8]
$$
    - Defina o tamanho da entrada da rede `max_len = 16`;
    - Defina as variáveis da função `maxlen = max_len+1` e `padding = "pre"`.
  

In [7]:
VOCAB_SIZE = 5000
MAX_LEN = 16

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=None, ## por que None?
    pad_to_max_tokens=True,
    output_mode="int",
    ragged=True,
)

vectorizer.adapt(texts)

vocab = np.array(vectorizer.get_vocabulary())

token_ids = vectorizer(texts).numpy()
token_ids[:1]



2026-06-03 20:03:58.459376: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Max
2026-06-03 20:03:58.459408: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-06-03 20:03:58.459413: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1780527838.459784 1462255 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1780527838.460012 1462255 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


array([array([   2, 1052,    8,   11,  125,    7,   79,  198, 1098,    5,  230,
              3770,   17,  160,  105,   11,  125, 2725,  198,  101,    6,   83,
              2295,   54,   16,  158,   24,   96, 1428,   58,    3, 1264,  250,
                54,   12,  156,    9,  235,  885,  114,   60, 3545,    7,  490,
               231,   43,    5,    2, 3686,    4,    1,  140,    4,  140,  172,
               158,   24,  946, 1098,    5,  648,    5,   82,  540,   42, 3545,
               775,   17,   24,   58,  688,   38,  125,   54,   36,  117, 1358,
                 1,  435,   42,   63,  707,    4, 1407, 4626,  250,   54,  889,
              3595,    4,  152,  158,   47,    7,  158,   21,   11,  125,  160,
              1113,   61,    6,  225,    8,  253, 2548,   10,  303,  175,   65,
                38,  410,    6, 3588,    3,  945, 1185,   12,   75,   15,   33,
               269,   28,  103,  158,    5,   38,   72, 3171,    5,    2, 1185,
                12,  118,   14,    4,  3

In [8]:
sequences = []
for ids in token_ids:
    for i in range(1, len(ids)):
        sequences.append(ids[0:i+1])
sequences[:2]


[array([   2, 1052]), array([   2, 1052,    8])]

In [9]:
sequences[-2:]

[array([  15,  796,   10,  400,    6,  106,   11,   19,   16,    1, 1102,
           2,   20,    7,   56,   52, 2055,  496,    6,  149,   23,   19,
        1150,   18,   11,   20,    7,    9,   59,   26, 1177,   17, 1848,
          44,  590,    2,   20,   14,    3, 1593,   94,   11,   27, 2249,
           3,  533,   18,    1,   10,  200,    1, 2356,   16,   23, 2200,
         490,    1,   13,   10,   28,  294,    2,   76,    1,  668,   21,
        2055,   10,  304,    9,  180,   15,    2, 2989,    7,   39, 4080,
         107, 2055,    7,   29,  321,   37,  973,   10,   90,   39,   23,
        1024,   22,   39,   23,  317,   27,    7,   22,    3, 3340,  321,
          35,    1,    1,   10,   90,   75,  410,    6, 1932, 2055,   16,
          23,  368,  143,   36,   23,    1,   13,  100,  792,   10,  410,
           6, 2207,  285,  353,   16,   11,  668,   34, 1165, 2378,   52,
        2610]),
 array([  15,  796,   10,  400,    6,  106,   11,   19,   16,    1, 1102,
           2,   20,   

In [83]:
MAX_LEN = 16
seqs = pad_sequences(sequences, maxlen=MAX_LEN+1, padding="pre")
#seqs = pad_sequences(sequences, maxlen=MAX_LEN+1, padding="post")

In [84]:
seqs[:5]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    2, 1052],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    2, 1052,    8],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    2, 1052,    8,   11],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    2, 1052,    8,   11,  125],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           2, 1052,    8,   11,  125,    7]], dtype=int32)

In [85]:
seqs.shape


(110949, 17)

### **4º Passo:**
<font size=3>

- Agora, iremos definir as variáveis $X$ e $y$ do modelo:
    - Onde $X$ serão as sequências até o **penúltimo elemento**, *e.g.*, $[ 0,\, 0,\, 0,\, 2]$;
    - e $y$ serão o **último elemento** das sequências, *e.g.*, $[5]$.
    <br>

- Verifique se $X$ apresenta dimensão `max_len`;


In [86]:
X = seqs[:, :-1]
y = seqs[:, -1]
X.shape, y.shape


((110949, 16), (110949,))

### **5º Passo:**
<font size=3>

- Desenvolva a rede neural:
    - Já sabemos do tamanho da camada de entrada, mas a **camada de saída** deverá *retornar qual o próximo token mais provável da lista de vocabulário*.
    - **Observação:** aqui, talvez, você ache incompatível a variável algo $y$ com a camada de saída, mas iremos dar um jeito na função de custo, a seguir.
      

In [87]:
EMBED_DIM = 32


In [89]:
x_in = layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input")
x = layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="embedding")(x_in)
x = layers.LSTM(128, return_sequences=False, name="lstm")(x)
x = layers.Dropout(0.2, name="dropout")(x)
x = layers.Dense(800, activation="relu", name="dense1")(x)
x_out = layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(x)

model = Model(x_in, x_out)
model.summary()

Model: "functional_24"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 16, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 800)            │       103,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5000)           │     4,005,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,350,632 (16.60 MB)

 Trainable params: 4,350,632 (16.60 MB)

 Non-trainable params: 0 (0.00 B)

### **6º Passo:**
<font size=3>

- Compile o modelo com:
    - Otimizador `Adam`;
    - Para lidar com **saídas probabilística** e **variável alvo categórica**, utilizaremos a **função de custo** [`sparse_categorical_crossentropy`](https://keras.io/api/losses/probabilistic_losses/#sparsecategoricalcrossentropy-class);
    - Métrica [`SparseTopKCategoricalAccuracy`](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/SparseTopKCategoricalAccuracy) para pegar os $k=5$ maiores valores da camada de saída.


In [90]:
model.compile(optimizer=Adam(learning_rate=1e-4), 
              loss="sparse_categorical_crossentropy", 
              metrics=[SparseTopKCategoricalAccuracy(k=5, name="SparseTopKCategoricalAccuracy")]
              )

### **7º Passo:**
<font size=3>

- Realize o **treinamento validativo** do modelo com:
    - 15 épocas (inicialmente);
    - `batch_size` > 100;
    - `validation_split = 0.2`.
      

In [91]:
model.fit(x=X, y=y, 
          epochs=15, 
          batch_size=64, 
          validation_split=0.2
          )




Epoch 1/15
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - SparseTopKCategoricalAccuracy: 0.2236 - loss: 6.4056 - val_SparseTopKCategoricalAccuracy: 0.2203 - val_loss: 6.2705
Epoch 2/15
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - SparseTopKCategoricalAccuracy: 0.2254 - loss: 6.1512 - val_SparseTopKCategoricalAccuracy: 0.2175 - val_loss: 6.2657
Epoch 3/15
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - SparseTopKCategoricalAccuracy: 0.2254 - loss: 6.1258 - val_SparseTopKCategoricalAccuracy: 0.2169 - val_loss: 6.2588
Epoch 4/15
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - SparseTopKCategoricalAccuracy: 0.2269 - loss: 6.0922 - val_SparseTopKCategoricalAccuracy: 0.2258 - val_loss: 6.2216
Epoch 5/15
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - SparseTopKCategoricalAccuracy: 0.2361 - loss: 6.0069 - val_SparseTopKCategoricalAccuracy: 0.2379 - val_loss: 6.1533
Epoch 6/15
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - SparseTopKCategoricalAccuracy: 0.2433 - loss: 5.9318 - val_SparseTopKCat

In [50]:
model.optimizer.learning_rate

<Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>

### **8º Passo:**
<font size=3>

Realize o **treinamento final** (sem validação) da melhor arquitetura que conseguiu modelar.

### **9º Passo:**
<font size=3>

- Defina uma função para **gerar** o próximo *token* dado uma entrada de texto:
    - O texto deve entrar na forma de *string*, *e.g.*, "the movie was", então a função deverá utilizar o objeto de `TextVectorization`, já definido, para obter os tokens indexados;
    - Em seguinda, o *array* de tokens indexados deverá passar pelo `pad_sequences`, com tamanho `max_len`;
    - Realize a predição do modelo e utilize a função `argmax` do `Numpy` para encontrar o índice do *token* mais provável;
    - Com este índice, use a lista de vocabulário para encontrar o *token* mais provável;
    - Anexe este token à sentença de entrada;
    - Coloque toda a estrutura de código dos itens acima em um *loop*, a fim de gerar as $n$ palavras mais provaveis. 
    <br>

- Seu modelo precisa completar ao menos **três palavras coerentes** com a frase de entrada.
      

In [111]:
def gerar_proximas_palavras(modelo, texto, max_len, vocab, n=3):
    # vetorizar o texto
    ids = vectorizer(texto).numpy().flatten().tolist()
    novas = [] # lista para armazenar as palavras geradas
    for _ in range(n):
        # padding para que o texto tenha o tamanho máximo
        x = pad_sequences([ids[-max_len:]], maxlen=max_len, padding="pre")
        # predizer as probabilidades de cada palavra
        probs = modelo.predict(x, verbose=0)[0]
        # não prever padding e UNK
        probs[0] = 0   # não prever padding
        probs[1] = 0   # não prever UNK
        # encontrar o índice da palavra mais provável
        next_id = int(np.argmax(probs))
        # adicionar o índice à lista de ids
        ids.append(next_id)
        # adicionar a palavra à lista de novas palavras
        novas.append(vocab[next_id])
    return novas

In [112]:
frases_teste = [
  "the movie was", 
  "we liked", 
  "the picture was", 
  "the music was", 
  "The overall opinion of the movie was"
]

for frase_original in frases_teste:
    print(f"Frase original: {frase_original}")
    frase = frase_original + " " + " ".join(gerar_proximas_palavras(model, frase_original, MAX_LEN, vocab, 3))
    print(f"Frase gerada: {frase}")
    print("-"*100)

print (f"vocab[1]: {vocab[1]}")

Frase original: the movie was
Frase gerada: the movie was the movie was
----------------------------------------------------------------------------------------------------
Frase original: we liked
Frase gerada: we liked to be is
----------------------------------------------------------------------------------------------------
Frase original: the picture was
Frase gerada: the picture was the movie was
----------------------------------------------------------------------------------------------------
Frase original: the music was
Frase gerada: the music was the movie was
----------------------------------------------------------------------------------------------------
Frase original: The overall opinion of the movie was
Frase gerada: The overall opinion of the movie was the movie was
----------------------------------------------------------------------------------------------------
vocab[1]: [UNK]
